In [1]:
# Loading the corrected features table.
import sys
sys.path.append("..")
import pandas as pd
import numpy as np
import xgboost as xgb

features_df = pd.read_parquet("../data/features.parquet")
features_df.shape

(2511, 11)

In [2]:
# Splitting into train and test by snapshot date, not randomly, since this respects time order.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [3]:
# Defining precision and recall at top k percent, matching how this model would be used.
def precision_at_k(y_true, y_scores, k_percent=10):
    """Return the precision among the top k percent highest scored companies."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    return y_true.iloc[top_k_idx].mean()

def recall_at_k(y_true, y_scores, k_percent=10):
    """Return the share of all actual failures captured within the top k percent by score."""
    n = int(len(y_true) * k_percent / 100)
    top_k_idx = np.argsort(y_scores)[-n:]
    caught = y_true.iloc[top_k_idx].sum()
    total_failures = y_true.sum()
    return caught / total_failures

In [4]:
# Rebuilding the baseline using days_since_last_accounts as the ranking score.
baseline_precision_at_10 = precision_at_k(
    test["is_failed"].reset_index(drop=True),
    test["days_since_last_accounts"].reset_index(drop=True),
    k_percent=10
)
baseline_recall_at_10 = recall_at_k(
    test["is_failed"].reset_index(drop=True),
    test["days_since_last_accounts"].reset_index(drop=True)
)
print(baseline_precision_at_10, baseline_recall_at_10)

0.54 0.16770186335403728


In [5]:
# Training XGBoost on the corrected, leakage-safe features.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [6]:
# Scoring the model with the corrected features.
model_scores = model.predict_proba(test[feature_cols])[:, 1]
model_precision_at_10 = precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
model_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
print(model_precision_at_10, model_recall_at_10)

0.6 0.18633540372670807


## Explainability with SHAP

The model above beats the baseline, but a risk score alone is not useful
to someone deciding what to do about a company. This section adds SHAP
values, so every prediction comes with a clear reason - which features
pushed the score up or down for that specific company.

In [10]:
!pip install shap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [shap]1/2 [shap]


In [11]:
# Explaining the model's predictions with SHAP values.
import shap

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(test[feature_cols])
shap_values.shape

(503, 8)

In [12]:
# Summarising average feature importance across all test companies.
import numpy as np

mean_abs_shap = np.abs(shap_values).mean(axis=0)
importance = pd.Series(mean_abs_shap, index=feature_cols).sort_values(ascending=False)
importance

company_age_years                     0.694467
filing_gap_missing                    0.393679
longest_filing_gap                    0.192528
days_since_last_accounts              0.174367
count_recent_resignations             0.085765
count_late_confirmation_statements    0.039398
count_new_charges                     0.022773
accounts_missing                      0.000000
dtype: float32

In [13]:
# Checking whether company age differs systematically between failed and live companies.
features_df.groupby("is_failed")["company_age_years"].describe()

,count,mean,std,min,25%,50%,75%,max
is_failed,,,,,,,,
0,1266.0,8.039640,10.970993,0.002738,1.062971,4.117728,10.784394,146.715948
1,1245.0,11.644101,13.775721,0.016427,4.065708,7.564682,13.245722,119.523614


### Explaining one company's prediction in plain language

In [14]:
# Looking at the SHAP explanation for one real company in the test set.
company_index = 0
company_number = test.iloc[company_index]["CompanyNumber"]
company_shap = shap_values[company_index]

explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
print("Company:", company_number)
print("Predicted risk score:", model_scores[company_index])
explanation

Company: 08670145
Predicted risk score: 0.61007196


company_age_years                     0.383267
days_since_last_accounts             -0.137613
filing_gap_missing                    0.128618
count_recent_resignations            -0.062752
longest_filing_gap                   -0.027221
count_late_confirmation_statements    0.016001
count_new_charges                    -0.014277
accounts_missing                      0.000000
dtype: float32

### Building a reusable plain language explanation function

In [15]:
# Turning SHAP values into a plain language explanation for one company.
def explain_prediction(company_shap: np.ndarray, feature_cols: list, top_n: int = 3) -> list[str]:
    """Return the top n features driving a prediction, as plain English sentences."""
    explanation = pd.Series(company_shap, index=feature_cols).sort_values(key=abs, ascending=False)
    sentences = []
    labels = {
        "company_age_years": "the company's age",
        "days_since_last_accounts": "how recently accounts were filed",
        "count_late_confirmation_statements": "late confirmation statements",
        "count_recent_resignations": "recent director resignations",
        "count_new_charges": "new charges registered against the company",
        "longest_filing_gap": "the longest gap between filings",
        "accounts_missing": "whether an accounts due date was on record",
        "filing_gap_missing": "whether the company had enough filing history",
    }
    for feature, value in explanation.head(top_n).items():
        direction = "increased" if value > 0 else "decreased"
        sentences.append(f"{labels.get(feature, feature)} {direction} the risk score")
    return sentences

In [16]:
# Testing the explanation function on the same example company.
explain_prediction(company_shap, feature_cols)

["the company's age increased the risk score",
 'how recently accounts were filed decreased the risk score',
 'whether the company had enough filing history increased the risk score']

In [18]:
# Loading one raw company file to check officer link structure.
import json
from pathlib import Path

sample_number = test.iloc[0]["CompanyNumber"]
sample_data = json.loads((Path("../data/raw") / f"{sample_number}.json").read_text())
sample_officer = sample_data["officers"]["items"][0]
sample_officer.get("links")

{'self': '/company/08670145/appointments/6PVzCuULdaIAGFU85bp1c_If0N4',
 'officer': {'appointments': '/officers/E4RrxMXusRZqynB7K7frSvWhIrQ/appointments'}}

In [20]:
# Loading the cohort to identify failed companies for the director network feature.
cohort = pd.read_parquet("../data/cohort.parquet")
failed = cohort[cohort["is_failed"] == 1]

In [21]:
# Counting how many unique director appointment links exist across failed companies.
director_links = set()
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    for o in data["officers"]["items"]:
        if o.get("officer_role") == "director":
            link = o.get("links", {}).get("officer", {}).get("appointments")
            if link:
                director_links.add(link)

len(director_links)

4426

In [22]:
# Collecting director appointment histories for a small test batch first.
import sys
sys.path.append("..")
from src.collect_directors import collect_directors

test_batch = set(list(director_links)[:5])
collect_directors(test_batch)

finished. collected: 5, skipped: 0, failed: 0


In [23]:
# Collecting appointment histories for all directors linked to failed companies.
collect_directors(director_links)

200/4426 processed, 200 collected, 0 skipped, 0 failed
400/4426 processed, 399 collected, 1 skipped, 0 failed
600/4426 processed, 599 collected, 1 skipped, 0 failed
800/4426 processed, 798 collected, 2 skipped, 0 failed
1000/4426 processed, 997 collected, 3 skipped, 0 failed
1200/4426 processed, 1197 collected, 3 skipped, 0 failed
1400/4426 processed, 1396 collected, 4 skipped, 0 failed
1600/4426 processed, 1596 collected, 4 skipped, 0 failed
1800/4426 processed, 1796 collected, 4 skipped, 0 failed
2000/4426 processed, 1996 collected, 4 skipped, 0 failed
2200/4426 processed, 2196 collected, 4 skipped, 0 failed
2400/4426 processed, 2396 collected, 4 skipped, 0 failed
2600/4426 processed, 2596 collected, 4 skipped, 0 failed
2800/4426 processed, 2796 collected, 4 skipped, 0 failed
3000/4426 processed, 2995 collected, 5 skipped, 0 failed
3200/4426 processed, 3195 collected, 5 skipped, 0 failed
3400/4426 processed, 3395 collected, 5 skipped, 0 failed
3600/4426 processed, 3595 collected, 5 s

In [24]:
# Looking at one director's appointment history to see the structure.
sample_director_file = list(Path("../data/directors").glob("*.json"))[0]
sample_appointments = json.loads(sample_director_file.read_text())
sample_appointments.get("items", [])[0] if sample_appointments.get("items") else "no items"

{'address': {'address_line_1': 'Lightowlers Lane',
  'country': 'England',
  'locality': 'Littleborough',
  'postal_code': 'OL15 0LN',
  'premises': 'Langdale'},
 'appointed_on': '2024-09-07',
 'appointed_to': {'company_name': 'KIRBYS (WHITBY) LIMITED',
  'company_number': '01456618',
  'company_status': 'active'},
 'name': 'Richard Anthony WHIPP',
 'country_of_residence': 'England',
 'is_pre_1992_appointment': False,
 'links': {'company': '/company/01456618'},
 'name_elements': {'forename': 'Richard',
  'title': 'Mr',
  'other_forenames': 'Anthony',
  'surname': 'WHIPP'},
 'nationality': 'British',
 'officer_role': 'director',
 'identity_verification_details': {'appointment_verification_statement_due_on': '2026-09-16'}}

## Building the director network feature

In [34]:
# Counting unique companies referenced across all collected director appointment histories.
other_companies = set()
for path in Path("../data/directors").glob("*.json"):
    data = json.loads(path.read_text())
    for item in data.get("items", []):
        number = item.get("appointed_to", {}).get("company_number")
        if number:
            other_companies.add(number)

len(other_companies)

22984

In [35]:
# Narrowing down to only companies currently showing a distress related status,
# since only these could ever contribute to the director distress signal.
DISTRESS_STATUSES = {"liquidation", "administration", "receivership", "voluntary-arrangement"}

distress_companies = set()
for path in Path("../data/directors").glob("*.json"):
    data = json.loads(path.read_text())
    for item in data.get("items", []):
        status = item.get("appointed_to", {}).get("company_status", "")
        number = item.get("appointed_to", {}).get("company_number")
        if number and any(d in status.lower() for d in DISTRESS_STATUSES):
            distress_companies.add(number)

len(distress_companies)

2576

In [36]:
# Testing the director links collector on a small batch first.
from src.collect_director_links import collect_director_links

test_batch = set(list(distress_companies)[:5])
collect_director_links(test_batch)

finished. collected: 5, skipped: 0, failed: 0


In [37]:
# Collecting filing history for all companies linked through director networks.
collect_director_links(distress_companies)

200/2576 processed, 198 collected, 2 skipped, 0 failed
400/2576 processed, 397 collected, 3 skipped, 0 failed
600/2576 processed, 597 collected, 3 skipped, 0 failed
800/2576 processed, 797 collected, 3 skipped, 0 failed
1000/2576 processed, 997 collected, 3 skipped, 0 failed
1200/2576 processed, 1197 collected, 3 skipped, 0 failed
1400/2576 processed, 1397 collected, 3 skipped, 0 failed
1600/2576 processed, 1597 collected, 3 skipped, 0 failed
1800/2576 processed, 1797 collected, 3 skipped, 0 failed
2000/2576 processed, 1997 collected, 3 skipped, 0 failed
2200/2576 processed, 2197 collected, 3 skipped, 0 failed
2400/2576 processed, 2397 collected, 3 skipped, 0 failed
finished. collected: 2571, skipped: 5, failed: 0


In [39]:
# Reloading the failure date function from Week 3.
def get_failure_date(filings_items):
    """Return the date liquidation started, or None if no clear marker exists."""
    for target_type in ("600", "COCOMP"):
        matches = [item["date"] for item in filings_items if item.get("type") == target_type]
        if matches:
            return min(matches)
    return None

In [40]:
# Computing failure dates for the linked companies, using the same logic from Week 3.
linked_failure_dates = {}
for path in Path("../data/director_links").glob("*.json"):
    number = path.stem
    data = json.loads(path.read_text())
    date = get_failure_date(data.get("items", []))
    if date:
        linked_failure_dates[number] = pd.to_datetime(date)

len(linked_failure_dates)

2077

In [41]:
# Rebuilding the director network feature using real failure dates, so nothing
# after the snapshot date can be counted.
def director_distress_count_safe(officer_link: str, snapshot_date: pd.Timestamp) -> int:
    """Count how many of a director's other companies had already failed,
    with a known failure date before the snapshot date.
    """
    officer_id = officer_link.split("/")[2]
    path = Path("../data/directors") / f"{officer_id}.json"
    if not path.exists():
        return 0
    data = json.loads(path.read_text())
    count = 0
    for item in data.get("items", []):
        other_number = item.get("appointed_to", {}).get("company_number")
        other_failure_date = linked_failure_dates.get(other_number)
        if other_failure_date is not None and other_failure_date < snapshot_date:
            count += 1
    return count


def company_director_distress_safe(officers: list, snapshot_date: pd.Timestamp) -> int:
    """Sum of safe distress counts across all directors of one company."""
    total = 0
    for o in officers:
        if o.get("officer_role") == "director":
            link = o.get("links", {}).get("officer", {}).get("appointments")
            if link:
                total += director_distress_count_safe(link, snapshot_date)
    return total

In [43]:
# Reloading cohort, failed companies, and failure dates needed for this test.
cohort = pd.read_parquet("../data/cohort.parquet")
failed = cohort[cohort["is_failed"] == 1]

failure_dates = {}
for number in failed["CompanyNumber"]:
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    date = get_failure_date(data["filings"]["items"])
    if date:
        failure_dates[number] = date

failure_df = pd.DataFrame({
    "CompanyNumber": list(failure_dates.keys()),
    "failure_date": pd.to_datetime(list(failure_dates.values())),
})
failure_df["snapshot_date"] = failure_df["failure_date"] - pd.DateOffset(months=12)

len(failure_df)

1253

In [44]:
# Testing the corrected feature on the same example company from before.
test_number = failure_df["CompanyNumber"].iloc[0]
test_snapshot = failure_df.loc[failure_df["CompanyNumber"] == test_number, "snapshot_date"].iloc[0]
test_data = json.loads((Path("../data/raw") / f"{test_number}.json").read_text())

company_director_distress_safe(test_data["officers"]["items"], test_snapshot)

0

In [45]:
# Rebuilding the director network feature across all failed companies, using real dates.
failed_director_scores_safe = {}
for _, row in failure_df.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    failed_director_scores_safe[number] = company_director_distress_safe(data["officers"]["items"], snapshot)

len(failed_director_scores_safe)

1253

In [46]:
# Reloading live companies with their snapshot dates for the safe director feature.
live_all = cohort[cohort["is_failed"] == 0].copy()
live_all["IncorporationDate"] = pd.to_datetime(live_all["IncorporationDate"], format="%d/%m/%Y")

rng = np.random.default_rng(1)
snapshot_pool = failure_df["snapshot_date"].to_numpy()

def sample_valid_snapshot(incorporation_date):
    valid = snapshot_pool[snapshot_pool > np.datetime64(incorporation_date)]
    if len(valid) == 0:
        return pd.NaT
    return rng.choice(valid)

live_all["snapshot_date"] = live_all["IncorporationDate"].apply(sample_valid_snapshot)
live_final = live_all.dropna(subset=["snapshot_date"]).copy()
len(live_final)

1266

In [47]:
# Building the safe director network feature across all live companies.
live_director_scores_safe = {}
for _, row in live_final.iterrows():
    number = row["CompanyNumber"]
    snapshot = row["snapshot_date"]
    data = json.loads((Path("../data/raw") / f"{number}.json").read_text())
    live_director_scores_safe[number] = company_director_distress_safe(data["officers"]["items"], snapshot)

len(live_director_scores_safe)

1266

In [48]:
# Merging the safe director network scores into the features table.
director_scores_safe = {**failed_director_scores_safe, **live_director_scores_safe}
features_df["director_distress_score_safe"] = features_df["CompanyNumber"].map(director_scores_safe)
features_df["director_distress_score_safe"].describe()

count    2511.000000
mean        0.148148
std         1.804696
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max        60.000000
Name: director_distress_score_safe, dtype: float64

In [49]:
# Redoing the train and test split now that the safe director feature is added.
features_df = features_df.sort_values("snapshot_date")
split_index = int(len(features_df) * 0.8)
split_date = features_df.iloc[split_index]["snapshot_date"]

train = features_df[features_df["snapshot_date"] < split_date]
test = features_df[features_df["snapshot_date"] >= split_date]

print(len(train), len(test), split_date)

2008 503 2025-01-21 00:00:00


In [50]:
# Checking which columns are currently in the features table.
features_df.columns.tolist()

['CompanyNumber',
 'snapshot_date',
 'days_since_last_accounts',
 'count_late_confirmation_statements',
 'count_recent_resignations',
 'count_new_charges',
 'company_age_years',
 'longest_filing_gap',
 'is_failed',
 'accounts_missing',
 'filing_gap_missing',
 'director_distress_score',
 'director_distress_score_safe']

In [51]:
# Dropping the leaky director score column now that the safe version replaces it.
features_df = features_df.drop(columns=["director_distress_score"])
features_df.columns.tolist()

['CompanyNumber',
 'snapshot_date',
 'days_since_last_accounts',
 'count_late_confirmation_statements',
 'count_recent_resignations',
 'count_new_charges',
 'company_age_years',
 'longest_filing_gap',
 'is_failed',
 'accounts_missing',
 'filing_gap_missing',
 'director_distress_score_safe']

In [52]:
# Retraining with the safe director network feature added.
feature_cols = [
    "days_since_last_accounts", "count_late_confirmation_statements",
    "count_recent_resignations", "count_new_charges", "company_age_years",
    "longest_filing_gap", "accounts_missing", "filing_gap_missing",
    "director_distress_score_safe",
]

model = xgb.XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05)
model.fit(train[feature_cols], train["is_failed"])

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,True
,eval_metric,None


In [53]:
# Scoring the model with the safe director network feature included.
model_scores = model.predict_proba(test[feature_cols])[:, 1]
model_precision_at_10 = precision_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
model_recall_at_10 = recall_at_k(test["is_failed"].reset_index(drop=True), pd.Series(model_scores))
print(model_precision_at_10, model_recall_at_10)

0.68 0.2111801242236025
